# Shot-by-Shot Video Analysis

This notebook performs shot detection, transcription, and optional VLM description on video files.

**Usage:**
1. Upload your video to Kaggle Datasets
2. Update the `VIDEO_PATH` in the Configuration cell
3. Run all cells

In [ ]:
# Install dependencies
!pip install scenedetect[opencv] pandas requests opencv-python-headless numpy

In [ ]:
# Import libraries
import os
import json
import requests
import pandas as pd
from scenedetect import detect, AdaptiveDetector
import subprocess
import tempfile
import base64
import cv2
import numpy as np

In [ ]:
# Configuration
WHISPER_CPP_PATH = "/usr/local/bin/whisper-cpp"  # Update path
OPENROUTER_API_KEY = ""  # Set your API key for VLM descriptions
VIDEO_PATH = "/kaggle/input/your-video/video.mp4"  # Update path to your video

In [ ]:
# Shot detection
def detect_shots(video_path, threshold=27.0):
    """Detect shots using adaptive detector."""
    scene_list = detect(video_path, AdaptiveDetector(adaptive_threshold=threshold))
    shots = []
    for idx, (start, end) in enumerate(scene_list):
        shots.append({
            "shot_id": idx + 1,
            "start_time": start.get_seconds(),
            "end_time": end.get_seconds()
        })
    return shots

shots = detect_shots(VIDEO_PATH)
print(f"Found {len(shots)} shots")
pd.DataFrame(shots)

In [ ]:
# Whisper transcription (if available)
def transcribe_video(video_path, whisper_path=WHISPER_CPP_PATH):
    """Transcribe video using whisper.cpp if available."""
    with tempfile.TemporaryDirectory() as tmpdir:
        cmd = [
            whisper_path,
            "-f", video_path,
            "-o", tmpdir,
            "--output-format", "json",
            "--compute-type", "int8"
        ]
        try:
            subprocess.run(cmd, check=True, capture_output=True)
            json_path = os.path.join(tmpdir, "output.json")
            with open(json_path, 'r') as f:
                data = json.load(f)
            return [{"text": s["text"].strip(), "start_time": s["start"], "end_time": s["end"]} 
                    for s in data.get("segments", [])]
        except:
            print("whisper.cpp not available, skipping transcription")
            return []

subtitles = transcribe_video(VIDEO_PATH)
print(f"Found {len(subtitles)} subtitle segments")

In [ ]:
# Merge results
def merge_shots_subtitles(shots, subtitles):
    """Merge shots with overlapping subtitles."""
    results = []
    for shot in shots:
        overlapping = [s for s in subtitles 
                      if s["start_time"] < shot["end_time"] and s["end_time"] > shot["start_time"]]
        subtitle_text = " ".join([s["text"] for s in overlapping]) if overlapping else ""
        results.append({
            "shot_id": shot["shot_id"],
            "start_time": shot["start_time"],
            "end_time": shot["end_time"],
            "subtitle": subtitle_text
        })
    return pd.DataFrame(results)

merged_df = merge_shots_subtitles(shots, subtitles)
merged_df

In [ ]:
# VLM description (optional)
def extract_frames(video_path, shot, num_frames=8):
    """Extract evenly-spaced frames from a shot."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    start_frame = int(shot["start_time"] * fps)
    end_frame = int(shot["end_time"] * fps)
    frame_indices = np.linspace(start_frame, end_frame, num_frames, dtype=int)
    
    frames_b64 = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            _, buffer = cv2.imencode('.jpg', frame)
            frames_b64.append(base64.b64encode(buffer).decode('utf-8'))
    cap.release()
    return frames_b64

def describe_frames(frames_b64, api_key, model="qwen/qwen-2.5-vl-7b-instruct:free"):
    """Get VLM description of frames."""
    content = [{"type": "text", "text": "Describe what happened in this video clip briefly."}]
    for f in frames_b64:
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{f}"}})
    
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": model, "messages": [{"role": "user", "content": content}], "max_tokens": 256},
        timeout=60
    )
    return response.json()["choices"][0]["message"]["content"]

# Run VLM if API key is set
if OPENROUTER_API_KEY:
    descriptions = []
    for shot in shots[:5]:  # Limit to first 5 for demo
        try:
            frames = extract_frames(VIDEO_PATH, shot)
            desc = describe_frames(frames, OPENROUTER_API_KEY)
            descriptions.append({"shot_id": shot["shot_id"], "description": desc})
            print(f"Shot {shot['shot_id']}: OK")
        except Exception as e:
            print(f"Shot {shot['shot_id']}: Failed - {e}")
    
    if descriptions:
        merged_df = merged_df.merge(pd.DataFrame(descriptions), on="shot_id", how="left")
        merged_df["description"] = merged_df["description"].fillna("")
else:
    print("No API key set, skipping VLM descriptions")

merged_df

In [ ]:
# Save output
output_path = "/kaggle/working/shot_by_shot_output.csv"
merged_df.to_csv(output_path, index=False)
print(f"Output saved to: {output_path}")

In [ ]:
# Download (in Kaggle)
from IPython.display import FileLink
FileLink(output_path)